In [ ]:
from dotenv import load_dotenv
load_dotenv()

## llm, document loading, document chunking, vector embeddings, vector database

In [64]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [ ]:
"""loader = PyPDFLoader("/Users/poojaveluri/Documents/genai/data/data_science_syllabus.pdf")
pdf_load = loader.load()
len(pdf_load)"""

In [77]:
loader = PyPDFLoader("/Users/poojaveluri/Documents/genai/data/medical_report.pdf")
pdf_load = loader.load()
len(pdf_load)

9

In [78]:
pdf_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
splitted_data = pdf_splitter.split_documents(pdf_load)
len(splitted_data)

26

In [79]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")

In [ ]:
# data_science syllabus pdf
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding= embeddings,
    persist_directory="./vector_rag_qna__db"
)

In [ ]:
# medical report pdf
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding= embeddings,
    persist_directory="./vector_rag_qna_med_report_db"
)

In [ ]:
query = "Machine learning and data science content"
result = vector_store.similarity_search(query = query)

context = ""
for doc in result:
    context += doc.page_content + "\n"

In [61]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash-lite")

## Chain - Content generate | prompt | llm | strparser

In [81]:
def get_content(query:str):
    result = vector_store.similarity_search(query = query)
    context = ""
    for doc in result:
        context += doc.page_content + "\n"
    return{
        "context":{context},
        "query":{query}
    }

In [66]:
prompt = PromptTemplate.from_template("""
You're a helpful assistant who provides answer to the user based on their query.
if you don't know the answer then say Sorry! i don't have enough information
Context : {context}
Question : {query}                                     
""")

In [67]:
rag_chain = get_content | prompt | llm

In [82]:
response = rag_chain.invoke("what is the patient name?")
print(response.text)

Based on the provided report, the patient's name is Ms. NIKITA CHUDHARY.


In [75]:
response = rag_chain.invoke("what is the duration of module 3?")
print(response.text)

The duration of Module 3 is 3 weeks.


In [ ]:
response = rag_chain.invoke("what is the content of java DSA?")
print(response.text)

Sorry! i don't have enough information


In [83]:
response = rag_chain.invoke("who referred these reports?")
print(response.text)

Based on the provided report, the test was referred by **DR NITIN NAHAR**.


In [84]:
response = rag_chain.invoke("what is the hemoglobic level?")
print(response.text)

Based on the provided report, the hemoglobin level is **13.10 g/dL** (with a biological reference interval of 12.00 - 15.00 g/dL).
